# X-bar & R 控制圖：計量型管制圖基礎

> **課程定位**：基本工具篇（2/6）  
> **前置課程**：[01_SPC_概論與變異分析](./01_SPC_概論與變異分析.ipynb)  
> **學習路徑**：01 概論 → **02 X-bar & R** → 03 I-MR → 04 P Chart → 05 製程能力 → 06 綜合案例

## 學習目標
- 理解 X-bar & R 圖的適用場景與公式推導
- 掌握控制圖常數表的使用方法
- 能夠手動計算並繪製 X-bar & R 圖
- 學會判讀 Western Electric Rules 偵測異常

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. X-bar & R 圖的適用場景

X-bar & R 圖是最常用的計量型控制圖，適用條件：

| 條件 | 說明 |
|------|------|
| 數據類型 | **連續型**（長度、重量、溫度等） |
| 取樣方式 | 每個時間點取 **多個樣本**（子組） |
| 子組大小 | **n = 2 ~ 10**（通常 4 或 5） |
| 假設 | 子組內數據近似常態分配 |

### 為什麼用「全距 R」而不用「標準差 S」？
- 當 n ≤ 10 時，全距 R 是標準差的高效估計量
- 計算簡單，適合現場快速手算
- 當 n > 10 時，應改用 X-bar & S 圖

## 2. 公式推導

### 基本計算

對於 $k$ 個子組，每組 $n$ 個樣本：

**子組平均：**
$$\bar{X}_i = \frac{1}{n} \sum_{j=1}^{n} X_{ij}$$

**子組全距：**
$$R_i = X_{i,\max} - X_{i,\min}$$

**總平均（Grand Mean）：**
$$\bar{\bar{X}} = \frac{1}{k} \sum_{i=1}^{k} \bar{X}_i$$

**平均全距：**
$$\bar{R} = \frac{1}{k} \sum_{i=1}^{k} R_i$$

### 管制界限

**X-bar 圖：**
$$UCL_{\bar{X}} = \bar{\bar{X}} + A_2 \bar{R}$$
$$LCL_{\bar{X}} = \bar{\bar{X}} - A_2 \bar{R}$$

**R 圖：**
$$UCL_R = D_4 \bar{R}$$
$$LCL_R = D_3 \bar{R}$$

其中 $A_2$、$D_3$、$D_4$ 為控制圖常數，取決於子組大小 $n$。

In [ ]:
# 控制圖常數表
constants = pd.DataFrame({
    'n': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'A2': [1.880, 1.023, 0.729, 0.577, 0.483, 0.419, 0.373, 0.337, 0.308],
    'D3': [0.000, 0.000, 0.000, 0.000, 0.000, 0.076, 0.136, 0.184, 0.223],
    'D4': [3.267, 2.574, 2.282, 2.114, 2.004, 1.924, 1.864, 1.816, 1.777],
    'd2': [1.128, 1.693, 2.059, 2.326, 2.534, 2.704, 2.847, 2.970, 3.078]
})
constants = constants.set_index('n')

print("=" * 50)
print("控制圖常數表（X-bar & R 圖）")
print("=" * 50)
display(constants)
print("\n說明：")
print("  A2: 用於計算 X-bar 圖的管制界限")
print("  D3, D4: 用於計算 R 圖的管制界限")
print("  d2: 用於從 R\u0304 估計 \u03c3（\u03c3\u0302 = R\u0304/d2）")

## 3. Step-by-Step 實作：半導體晶圓厚度

使用 Notebook 01 中的晶圓厚度情境，重新生成相同數據（相同 seed），逐步建立 X-bar & R 圖。

In [ ]:
# 重新生成晶圓數據（與 Notebook 01 相同 seed）
np.random.seed(42)
n_subgroups = 25
subgroup_size = 5

wafer_data = np.random.normal(loc=500, scale=2, size=(n_subgroups, subgroup_size))

# Step 1: 計算子組統計量
xbar = wafer_data.mean(axis=1)  # 子組平均
R = wafer_data.max(axis=1) - wafer_data.min(axis=1)  # 子組全距

# Step 2: 計算總平均與平均全距
xbar_bar = xbar.mean()  # 總平均 (X\u034f)
R_bar = R.mean()         # 平均全距 (R\u0304)

# Step 3: 查表取得常數 (n=5)
A2 = 0.577
D3 = 0.000
D4 = 2.114

# Step 4: 計算管制界限
UCL_xbar = xbar_bar + A2 * R_bar
LCL_xbar = xbar_bar - A2 * R_bar
UCL_R = D4 * R_bar
LCL_R = D3 * R_bar

print("=" * 50)
print("X-bar & R 圖計算結果")
print("=" * 50)
print(f"\n\U0001f4ca 基本統計量：")
print(f"  總平均 (X\u034f)  = {xbar_bar:.4f} \u03bcm")
print(f"  平均全距 (R\u0304) = {R_bar:.4f} \u03bcm")
print(f"\n\U0001f4cf X-bar 圖管制界限：")
print(f"  UCL = {xbar_bar:.4f} + {A2} \u00d7 {R_bar:.4f} = {UCL_xbar:.4f}")
print(f"  CL  = {xbar_bar:.4f}")
print(f"  LCL = {xbar_bar:.4f} - {A2} \u00d7 {R_bar:.4f} = {LCL_xbar:.4f}")
print(f"\n\U0001f4cf R 圖管制界限：")
print(f"  UCL = {D4} \u00d7 {R_bar:.4f} = {UCL_R:.4f}")
print(f"  CL  = {R_bar:.4f}")
print(f"  LCL = {D3} \u00d7 {R_bar:.4f} = {LCL_R:.4f}")

# 估計製程標準差
d2 = 2.326
sigma_hat = R_bar / d2
print(f"\n\U0001f4d0 製程標準差估計：\u03c3\u0302 = R\u0304/d2 = {R_bar:.4f}/{d2} = {sigma_hat:.4f} \u03bcm")

In [ ]:
def plot_xbar_r(data_2d, subgroup_labels=None, title="X-bar & R 控制圖"):
    """
    繪製 X-bar & R 控制圖
    
    Parameters:
    -----------
    data_2d : np.array, shape (k, n)
        k 個子組，每組 n 個樣本
    subgroup_labels : list, optional
        子組標籤
    title : str
        圖表標題
    """
    k, n = data_2d.shape
    
    # 計算統計量
    xbar = data_2d.mean(axis=1)
    R = data_2d.max(axis=1) - data_2d.min(axis=1)
    xbar_bar = xbar.mean()
    R_bar = R.mean()
    
    # 常數表
    A2_table = {2: 1.880, 3: 1.023, 4: 0.729, 5: 0.577, 6: 0.483, 
                7: 0.419, 8: 0.373, 9: 0.337, 10: 0.308}
    D3_table = {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0.076, 8: 0.136, 9: 0.184, 10: 0.223}
    D4_table = {2: 3.267, 3: 2.574, 4: 2.282, 5: 2.114, 6: 2.004, 
                7: 1.924, 8: 1.864, 9: 1.816, 10: 1.777}
    
    A2 = A2_table[n]
    D3 = D3_table[n]
    D4 = D4_table[n]
    
    # 管制界限
    UCL_x = xbar_bar + A2 * R_bar
    LCL_x = xbar_bar - A2 * R_bar
    UCL_r = D4 * R_bar
    LCL_r = D3 * R_bar
    
    x_axis = range(1, k + 1)
    if subgroup_labels is None:
        subgroup_labels = x_axis
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    
    # --- X-bar 圖 ---
    ooc_x = (xbar > UCL_x) | (xbar < LCL_x)
    ax1.plot(x_axis, xbar, 'b-o', markersize=5, label='子組平均', zorder=3)
    if ooc_x.any():
        ax1.scatter(np.array(list(x_axis))[ooc_x], xbar[ooc_x], 
                    color='red', s=100, zorder=5, label='超出管制界限', marker='s')
    ax1.axhline(y=xbar_bar, color='green', linewidth=2, label=f'CL = {xbar_bar:.3f}')
    ax1.axhline(y=UCL_x, color='red', linestyle='--', linewidth=1.5, label=f'UCL = {UCL_x:.3f}')
    ax1.axhline(y=LCL_x, color='red', linestyle='--', linewidth=1.5, label=f'LCL = {LCL_x:.3f}')
    ax1.set_ylabel('X-bar (子組平均)', fontsize=12)
    ax1.set_title('X-bar 圖', fontsize=13)
    ax1.legend(loc='upper right', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # --- R 圖 ---
    ooc_r = (R > UCL_r) | (R < LCL_r)
    ax2.plot(x_axis, R, 'r-s', markersize=5, label='子組全距', zorder=3)
    if ooc_r.any():
        ax2.scatter(np.array(list(x_axis))[ooc_r], R[ooc_r],
                    color='darkred', s=100, zorder=5, label='超出管制界限', marker='D')
    ax2.axhline(y=R_bar, color='green', linewidth=2, label=f'CL = {R_bar:.3f}')
    ax2.axhline(y=UCL_r, color='red', linestyle='--', linewidth=1.5, label=f'UCL = {UCL_r:.3f}')
    if LCL_r > 0:
        ax2.axhline(y=LCL_r, color='red', linestyle='--', linewidth=1.5, label=f'LCL = {LCL_r:.3f}')
    ax2.set_xlabel('子組編號', fontsize=12)
    ax2.set_ylabel('R (全距)', fontsize=12)
    ax2.set_title('R 圖', fontsize=13)
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 回傳結果
    return {
        'xbar': xbar, 'R': R,
        'xbar_bar': xbar_bar, 'R_bar': R_bar,
        'UCL_x': UCL_x, 'LCL_x': LCL_x,
        'UCL_r': UCL_r, 'LCL_r': LCL_r,
        'ooc_xbar': np.where(ooc_x)[0] + 1,
        'ooc_R': np.where(ooc_r)[0] + 1
    }

# 繪製晶圓厚度的 X-bar & R 圖
result = plot_xbar_r(wafer_data, title="半導體晶圓厚度 X-bar & R 控制圖")

print(f"\nX-bar 圖超出管制界限的子組: {result['ooc_xbar'] if len(result['ooc_xbar']) > 0 else '無'}")
print(f"R 圖超出管制界限的子組: {result['ooc_R'] if len(result['ooc_R']) > 0 else '無'}")

## 4. 判讀規則：Western Electric Rules

控制圖不僅檢查「點是否超出界限」，還有系統性的判讀規則：

### 四大基本規則

| 規則 | 名稱 | 條件 | 可能原因 |
|------|------|------|----------|
| Rule 1 | 超出管制界限 | 任一點超出 \u00b13\u03c3 | 突發異常事件 |
| Rule 2 | 連串（Run） | 連續 7 點在中心線同側 | 製程平均偏移 |
| Rule 3 | 趨勢（Trend） | 連續 6 點持續上升或下降 | 工具磨損、溫度漂移 |
| Rule 4 | 接近界限 | 3 點中有 2 點超出 \u00b12\u03c3 | 製程變異增大 |

> **知識補給站**  
> 這些規則最早由 Western Electric 公司的統計品管手冊提出，因此得名。在實務中，Rule 1 和 Rule 2 是最常用的。

In [ ]:
def check_western_electric_rules(values, cl, ucl, lcl):
    """
    檢查 Western Electric Rules
    
    Returns: dict of rule violations {rule_name: [violating_indices]}
    """
    n = len(values)
    sigma = (ucl - cl) / 3  # 從管制界限反推 sigma
    
    violations = {
        'Rule 1 (超出3\u03c3)': [],
        'Rule 2 (連續7點同側)': [],
        'Rule 3 (連續6點趨勢)': [],
        'Rule 4 (3中2超出2\u03c3)': []
    }
    
    # Rule 1: 超出管制界限
    for i in range(n):
        if values[i] > ucl or values[i] < lcl:
            violations['Rule 1 (超出3\u03c3)'].append(i)
    
    # Rule 2: 連續 7 點在同一側
    for i in range(n - 6):
        window = values[i:i+7]
        if all(w > cl for w in window) or all(w < cl for w in window):
            violations['Rule 2 (連續7點同側)'].extend(range(i, i+7))
    violations['Rule 2 (連續7點同側)'] = sorted(set(violations['Rule 2 (連續7點同側)']))
    
    # Rule 3: 連續 6 點持續上升或下降
    for i in range(n - 5):
        window = values[i:i+6]
        diffs = np.diff(window)
        if all(d > 0 for d in diffs) or all(d < 0 for d in diffs):
            violations['Rule 3 (連續6點趨勢)'].extend(range(i, i+6))
    violations['Rule 3 (連續6點趨勢)'] = sorted(set(violations['Rule 3 (連續6點趨勢)']))
    
    # Rule 4: 3 點中有 2 點超出 \u00b12\u03c3
    two_sigma_upper = cl + 2 * sigma
    two_sigma_lower = cl - 2 * sigma
    for i in range(n - 2):
        window = values[i:i+3]
        beyond_2sigma = sum(1 for w in window if w > two_sigma_upper or w < two_sigma_lower)
        if beyond_2sigma >= 2:
            violations['Rule 4 (3中2超出2\u03c3)'].extend(range(i, i+3))
    violations['Rule 4 (3中2超出2\u03c3)'] = sorted(set(violations['Rule 4 (3中2超出2\u03c3)']))
    
    return violations

# 對晶圓數據檢查
violations = check_western_electric_rules(result['xbar'], result['xbar_bar'], 
                                          result['UCL_x'], result['LCL_x'])

print("=" * 50)
print("Western Electric Rules 檢查結果（X-bar 圖）")
print("=" * 50)
for rule, indices in violations.items():
    if indices:
        print(f"\n\u26a0\ufe0f  {rule}:")
        print(f"   違規子組: {[i+1 for i in indices]}")
    else:
        print(f"\n\u2705 {rule}: 無違規")

## 5. 實務案例：CNC 加工件外徑

### 情境描述

某 CNC 車床加工精密零件外徑，規格要求 25.00 mm。
- 每小時取 **4 件** 量測
- 收集 **30 個子組**
- 在第 20 組時，刀具發生微量磨損，造成**平均偏移**

In [ ]:
np.random.seed(2024)

n_sub = 30
n_size = 4

# 前 20 組正常，後 10 組平均偏移 +0.03mm
normal_data = np.random.normal(loc=25.00, scale=0.02, size=(20, n_size))
shifted_data = np.random.normal(loc=25.03, scale=0.02, size=(10, n_size))
cnc_data = np.concatenate([normal_data, shifted_data], axis=0)

print("CNC 加工件外徑數據（模擬）")
print(f"子組數: {n_sub}, 子組大小: {n_size}")
print(f"前 20 組: \u03bc=25.00mm, \u03c3=0.02mm")
print(f"後 10 組: \u03bc=25.03mm, \u03c3=0.02mm（刀具磨損導致偏移）")
print()

# 繪製 X-bar & R 圖
cnc_result = plot_xbar_r(cnc_data, title="CNC 加工件外徑 X-bar & R 控制圖")

# 檢查 Western Electric Rules
print("\n" + "=" * 50)
print("Western Electric Rules 檢查")
print("=" * 50)
v = check_western_electric_rules(cnc_result['xbar'], cnc_result['xbar_bar'],
                                  cnc_result['UCL_x'], cnc_result['LCL_x'])
for rule, indices in v.items():
    if indices:
        print(f"\n\u26a0\ufe0f  {rule}: 子組 {[i+1 for i in indices]}")
    else:
        print(f"\u2705 {rule}: 無違規")

### 案例解讀

從上圖可以觀察到：
1. **X-bar 圖**：第 20 組之後，多個點超出或接近 UCL → 製程平均已偏移
2. **R 圖**：全距保持穩定 → 製程**變異性**未改變，問題在於**平均值**偏移

**製程工程師的行動建議：**
- 檢查刀具磨損狀況，安排換刀
- 對第 20 組之後的產品進行全數檢驗
- 調查是否有其他因素（溫度、材料）同時改變

> **知識補給站**  
> X-bar 圖偵測**平均值的變化**，R 圖偵測**變異性的變化**。兩張圖要一起看：先看 R 圖確認變異穩定，再看 X-bar 圖判斷平均值。

## 6. 本章小結

| 項目 | 內容 |
|------|------|
| 適用場景 | 連續型數據，子組大小 n=2~10 |
| 核心公式 | UCL/LCL = X\u034f \u00b1 A\u2082R\u0304（X-bar 圖）；D\u2083R\u0304 / D\u2084R\u0304（R 圖） |
| 判讀順序 | 先看 R 圖 → 再看 X-bar 圖 |
| 判讀規則 | Western Electric Rules（超限、連串、趨勢、接近界限） |
| 關鍵函數 | `plot_xbar_r()`、`check_western_electric_rules()` |

---

## 課堂練習

### 基礎題
1. 給定以下 5 個子組（n=3）的數據，手動計算 X\u034f、R\u0304、UCL、LCL：
   - 子組 1: [10.2, 10.5, 10.3]
   - 子組 2: [10.1, 10.4, 10.6]
   - 子組 3: [10.3, 10.2, 10.5]
   - 子組 4: [10.4, 10.3, 10.1]
   - 子組 5: [10.5, 10.6, 10.2]

### 進階題
2. 使用 `plot_xbar_r()` 繪製上述數據的控制圖，並用程式驗證手算結果。

### 挑戰題
3. 修改 `check_western_electric_rules()` 函數，新增 **Rule 5**：連續 14 點交替上下波動（鋸齒狀）。此規則通常表示量測系統問題或兩個交替的原料來源。

---

> **下一堂課**：[03_個別值管制圖_I-MR](./03_個別值管制圖_I-MR.ipynb) -- 當每次只有一個量測值時怎麼辦？